## 23. Stress-Testing the Loss Distribution

Combining Section 18's stress scenarios with the copula simulation: re-run the full Monte Carlo loss simulation under each scenario's stressed PD, rather than just scaling the point-estimate Expected Loss.

In [ ]:
# ============================================================
# CELL — Re-run the Monte Carlo simulation under stress scenarios
# ============================================================
"""
"""

# Section 18 established: PD_stressed = PD_baseline * (unrate_hr ** unemployment_delta)
# We reuse that same multiplicative shock, applied to sim_pd, then re-run the
# full copula simulation under each scenario's stressed PD.

stress_scenarios_copula = {
    'Baseline':         {'unemployment_delta': 0.0},
    'Adverse':          {'unemployment_delta': 3.0},
    'Severely Adverse': {'unemployment_delta': 6.0},
}

ASSUMED_UNRATE_HR = 1.12  # same literature-substituted value from Section 18

def run_copula_simulation(pd_array, lgd_array, ead_array, rho_val, n_sim, batch_size=500, seed=42):
    np.random.seed(seed)
    n = len(pd_array)
    threshold = norm.ppf(np.clip(pd_array, 1e-6, 1 - 1e-6))
    sqrt_rho = np.sqrt(rho_val)
    sqrt_1mr = np.sqrt(1 - rho_val)
    n_batches = int(np.ceil(n / batch_size))
    losses = np.zeros(n_sim)

    for sim in range(n_sim):
        Z = np.random.standard_normal()
        total = 0.0
        for b in range(n_batches):
            start = b * batch_size
            end = min(start + batch_size, n)
            epsilon = np.random.standard_normal(end - start)
            asset_value = sqrt_rho * Z + sqrt_1mr * epsilon
            defaulted = asset_value < threshold[start:end]
            total += np.where(defaulted, lgd_array[start:end] * ead_array[start:end], 0.0).sum()
        losses[sim] = total
    return losses

stress_loss_distributions = {}

for name, shock in stress_scenarios_copula.items():
    multiplier = ASSUMED_UNRATE_HR ** shock['unemployment_delta']
    stressed_pd = np.clip(sim_pd * multiplier, 0, 1)

    print(f"Running {name} scenario (PD multiplier = {multiplier:.4f})...")
    losses = run_copula_simulation(stressed_pd, sim_lgd, sim_ead, rho, N_SIM, seed=42)
    stress_loss_distributions[name] = losses
    print(f"  Mean loss: ${losses.mean():,.0f}\n")

print("All stress scenarios simulated.")

In [ ]:
# ============================================================
# CELL — Compare VaR/CVaR and distributions across scenarios
# ============================================================
comparison_rows = []
for name, losses in stress_loss_distributions.items():
    sorted_l = np.sort(losses)
    v95, c95 = compute_var_cvar(sorted_l, 0.95)
    v999, c999 = compute_var_cvar(sorted_l, 0.999)
    comparison_rows.append({
        'scenario': name,
        'mean_loss': losses.mean(),
        'var_95': v95,
        'cvar_95': c95,
        'var_999': v999,
        'cvar_999': c999,
    })

stress_comparison_df = pd.DataFrame(comparison_rows).set_index('scenario')
print("--- VaR/CVaR Across Stress Scenarios ---")
print(stress_comparison_df.round(0))

fig, ax = plt.subplots(figsize=(13, 7))
colors = {'Baseline': '#2ecc71', 'Adverse': '#f39c12', 'Severely Adverse': '#e74c3c'}
for name, losses in stress_loss_distributions.items():
    ax.hist(losses, bins=60, alpha=0.5, label=name, color=colors[name])
ax.set_title('Loss Distribution Shift Under Stress Scenarios')
ax.set_xlabel('Portfolio Loss ($)')
ax.set_ylabel('Number of Scenarios')
ax.legend()
plt.tight_layout()
plt.show()